# Colab runner — limited-data pneumonia classification

Runner for training on Colab Pro: mounts Drive, clones this (private) repo, installs the
pinned packages, and checks the environment. No project logic lives here - everything is
called from `src/` and the root scripts.

## Before the first run

1. **Runtime → Change runtime type → L4 or T4 GPU.** Not A100 - these models are small, so
   an A100 burns compute units much faster without a speedup that matters here.
2. In Drive, create `MyDrive/pneumonia-project/data/`. The notebook creates the rest itself.
3. Add a GitHub token as a Colab secret named `GITHUB_TOKEN` - the key icon in the left
   sidebar, with notebook access switched on. It must be a **classic** token with the `repo`
   scope: the repo sits under Or's account, and fine-grained tokens only reach
   repositories owned by the account that issued them. The repo is private, so the clone
   below fails without it.

## Where things live
| Path | Holds | Survives the session |
|---|---|---|
| `MyDrive/pneumonia-project/` | dataset zip, results, checkpoints | yes |
| `/content/limited-data-pneumonia-classification/` | this repo, cloned fresh | no |
| `/content/data/raw/chest_xray/` | the dataset, unzipped | no |

Drive is slow with many small files, so the dataset lives there as a single zip and is
unpacked onto the session's local disk. Anything worth keeping is written back to Drive.

## 1. GPU check

Should print a T4 or L4 and `cuda available: True`. If it doesn't, fix the runtime before
running anything below.

In [ ]:
# First cell that runs anything, deliberately: on a CPU-only runtime every cell below still
# succeeds, and the problem only shows up much later as unusably slow training.

# A leading "!" sends the line to the shell instead of Python. nvidia-smi is NVIDIA's driver
# utility; -L lists just the attached GPUs, one line each, instead of its full status table.
!nvidia-smi -L

# torch is importable already - Colab ships a CUDA-matched build preinstalled. Keeping this
# import here rather than at the top of the notebook means the cell stands on its own, which
# matters because it runs before the pip install a few cells down.
import torch

# is_available() is False in two different situations: no GPU attached, or a torch build with
# no CUDA support. Both are fatal here, and the second is exactly what a careless pip install
# causes - which is why this same line gets checked again after the install.
print(f"cuda available: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    # get_device_name(0) returns the marketing name of GPU number 0, e.g. "Tesla T4".
    print(f"gpu: {torch.cuda.get_device_name(0)}")
    # torch.version.cuda is the CUDA version this torch was *compiled against*, which is the
    # number that has to keep matching the driver above.
    print(f"torch: {torch.__version__} (CUDA {torch.version.cuda})")
else:
    print("No GPU. Runtime -> Change runtime type -> T4 or L4, then rerun this cell.")

## 2. Mount Google Drive

Opens an authorisation dialog the first time in a session. Nothing is printed on purpose:
notebook outputs get committed to the repo, so this cell shouldn't list your Drive contents.

In [ ]:
from google.colab import drive

# mount() makes Google Drive appear as an ordinary folder under /content/drive, so the rest of
# the notebook can treat it as normal paths. Reconnects silently on later runs in the same
# session, so this cell is safe to re-run.
drive.mount("/content/drive")

## 3. Paths

Every path the notebook uses, defined once here. Creates the Drive output folders and stops
with a clear message if the project folder itself is missing.

In [ ]:
from pathlib import Path
import os

# --- persistent, on Drive ---
DRIVE_ROOT = Path("/content/drive/MyDrive/pneumonia-project")
DATA_ZIP = DRIVE_ROOT / "data" / "chest_xray.zip"
RESULTS_METRICS = DRIVE_ROOT / "results" / "metrics"
RESULTS_FIGURES = DRIVE_ROOT / "results" / "figures"
CHECKPOINTS     = DRIVE_ROOT / "checkpoints"

# --- ephemeral, on the session disk (wiped when the runtime ends) ---
REPO_DIR  = Path("/content/limited-data-pneumonia-classification")
DATA_DIR  = Path("/content/data/raw")          # the zip unpacks into here
DATA_ROOT = DATA_DIR / "chest_xray"            # ...producing this
# src.paths.data_root() reads this variable to find the dataset. Setting it here in the kernel
# covers the `!python ...` cells too: a shell command launched from a notebook inherits the
# kernel's environment variables.
os.environ["PNEUMONIA_DATA_ROOT"] = str(DATA_ROOT)

# Checkpoints, metrics and figures go to Drive, not the session disk - src.paths.output_root()
# reads this, so run_experiment.py and run_all.py write there without being told each time.
os.environ["PNEUMONIA_OUTPUT_ROOT"] = str(DRIVE_ROOT)

# Throwaway runs (2-epoch smoke tests) write here instead, so they cannot mark a real run
# finished and get it skipped later.
SMOKE_ROOT = Path("/content/smoke")


# Owner/name only, no "https://" and no ".git": the clone cell has to splice a token into the
# middle of the URL, so it builds the full thing from this rather than editing a finished URL.
REPO_SLUG = "orbakish/limited-data-pneumonia-classification"

# The branch this session runs: "main", unless an unmerged branch is being tested on Colab.
REPO_BRANCH = "main"


# Stop here, not three cells later, if the folder from the header cell was never created.
assert DRIVE_ROOT.is_dir(), f"Create {DRIVE_ROOT} in Drive first - see the header cell."

for folder in (RESULTS_METRICS, RESULTS_FIGURES, CHECKPOINTS):
    # parents=True also creates the missing folders on the way (results/ before metrics/);
    # exist_ok=True turns a re-run into a no-op instead of an error.
    folder.mkdir(parents=True, exist_ok=True)
    print(f"ready: {folder}")

## 4. GitHub token

Read from Colab Secrets so the token never appears in this file. The fallback prompt is there
for a runtime where the secret isn't set up — it has to be retyped every session.

In [ ]:
from getpass import getpass

try:
    from google.colab import userdata
    # userdata.get reads a secret from the key-icon panel in the sidebar. It raises if the
    # secret doesn't exist, or if "Notebook access" is switched off for this notebook.
    GITHUB_TOKEN = userdata.get("GITHUB_TOKEN")
    print("token: loaded from Colab secrets")
except Exception as error:
    # Deliberately broad: Colab raises several different secret-related errors, all of which
    # mean the same thing here - fall back to asking. getpass reads a line from an input box
    # without echoing it, so the token stays out of the cell output that gets committed.
    print(f"token: secret unavailable ({type(error).__name__}), falling back to a prompt")
    GITHUB_TOKEN = getpass("GitHub token (classic, repo scope): ")

# Never print the token. Its length is enough to catch the usual mistakes: an empty box, or a
# value pasted with the surrounding quotes still attached.
print(f"token: {len(GITHUB_TOKEN)} characters")

## 5. Clone or update the repo

Clones on the first run of a session and pulls afterwards, then prints the branch and commit
so results can always be traced back to the code that produced them.

In [ ]:
import os
import subprocess
import sys


def run_git(*args):
    """Run a git command in the cloned repo, with the token blanked from anything it prints.

    Returns git's stdout as a stripped string. Raises RuntimeError on a non-zero exit,
    with the token replaced by *** in the message.
    """
    # capture_output=True collects stdout and stderr instead of letting them stream to the
    # cell; text=True returns them as str rather than raw bytes.
    result = subprocess.run(["git", *args], capture_output=True, text=True)
    # git echoes the remote URL - token included - in most of its error messages, and cell
    # output is committed along with the notebook, so every message is scrubbed first.
    if result.returncode != 0:
        raise RuntimeError(result.stderr.replace(GITHUB_TOKEN, "***").strip())
    return result.stdout.strip()


# Built here and never stored in a constant: anything that gets a name tends to get printed.
# git does write this URL into .git/config, but /content dies with the session, so it goes too.
auth_url = f"https://{GITHUB_TOKEN}@github.com/{REPO_SLUG}.git"

if REPO_DIR.is_dir():
    # -C <dir> runs git as if it had been started inside that directory, which avoids having
    # to change the notebook's working directory just to reach the repo.
    # fetch first: a branch pushed after this session's clone is unknown to the local copy, and
    # checkout would fail with "did not match any file(s) known to git".
    run_git("-C", str(REPO_DIR), "fetch", "--quiet", "--prune")
    run_git("-C", str(REPO_DIR), "checkout", "--quiet", REPO_BRANCH)
    run_git("-C", str(REPO_DIR), "pull", "--quiet")
    print("repo: pulled")
else:
    # --branch picks the branch to check out. Without it the clone lands on the repo's default
    # branch, so testing an unmerged branch would silently run main's code instead.
    run_git("clone", "--quiet", "--branch", REPO_BRANCH, auth_url, str(REPO_DIR))
    print("repo: cloned")



# Work from inside the repo so `!python check_env.py` and the entry points later resolve their
# relative paths, and so `import src...` finds the cloned package.
os.chdir(REPO_DIR)
if str(REPO_DIR) not in sys.path:
    # sys.path is the list of directories import searches; inserting at 0 puts the repo ahead
    # of Colab's own site-packages, so a name clash resolves to our code.
    sys.path.insert(0, str(REPO_DIR))

# Pin down exactly which commit this session ran - otherwise results produced weeks later are
# impossible to tie back to the code that produced them.
branch = run_git("-C", str(REPO_DIR), "rev-parse", "--abbrev-ref", "HEAD")
commit = run_git("-C", str(REPO_DIR), "rev-parse", "--short", "HEAD")
print(f"repo: {branch} @ {commit}")

## 6. Install requirements

Prints the torch version before and after. They must match: `requirements.txt` leaves torch,
torchvision and numpy unpinned precisely so pip leaves Colab's CUDA builds alone.

In [ ]:
# The most dangerous cell in the notebook. requirements.txt leaves torch, torchvision and numpy
# without versions on purpose: pip treats an already-satisfied bare requirement as a no-op, so
# Colab's CUDA-matched builds survive untouched. A pinned version - or --upgrade - makes pip
# replace them with generic PyPI wheels, and GPU support disappears with no error anywhere.

print(f"before: torch {torch.__version__} (CUDA {torch.version.cuda})")

# -q trims the install log to the lines worth reading. NEVER add --upgrade here.
# (Worth knowing: IPython substitutes {name} in a "!" line with the Python variable of that
# name, so any shell line containing braces - an f-string, say - will not survive this route.)
!pip install -q -r requirements.txt

# The kernel is still holding the torch it imported back in cell 3, so asking it again would
# just repeat the old numbers. A separate interpreter imports whatever is on disk *now*.
probe = "import torch; print(torch.__version__, torch.version.cuda, torch.cuda.is_available())"
# sys.executable is the full path of the Python running this kernel, so the subprocess is
# guaranteed to be the same interpreter with the same installed packages.
result = subprocess.run([sys.executable, "-c", probe], capture_output=True, text=True)
assert result.returncode == 0, result.stderr

version, cuda, available = result.stdout.split()
print(f"after:  torch {version} (CUDA {cuda}), cuda available: {available}")

assert (version, cuda) == (torch.__version__, torch.version.cuda), \
    "pip replaced Colab's torch. Stop here - restart the runtime and check requirements.txt."

## 7. Environment check

Runs the repo's own `check_env.py`. Expect a `cuda` device line and three models each printing
`output=(1, 1000)`.

In [ ]:
# The Colab half of the smoke test. Calling the committed script instead of repeating its logic
# here keeps this notebook a runner, and means local and Colab are verified by the same code.
# The working directory is the repo, set by the clone cell, so the bare filename resolves.
!python check_env.py

## 8. Drive write test

Writes a small file into the results folder, reads it back, deletes it. Better to find a
permissions or quota problem now than after an hour of GPU time.


In [ ]:
import json

probe_file = RESULTS_METRICS / "_write_test.json"
# write_text() creates the file and writes a string in one call, replacing any existing content.
probe_file.write_text(json.dumps({"ok": True}))

# Reading it back is the part that matters. Drive is a network filesystem behind a local-looking
# path, so a write can be accepted into a buffer and still never reach Drive.
assert json.loads(probe_file.read_text()) == {"ok": True}

# unlink() is pathlib's delete - no reason to leave a test artefact in the results tree.
probe_file.unlink()
print(f"Drive write/read/delete OK: {RESULTS_METRICS}")

## 9. Unpack the dataset
Needs `chest_xray.zip` in `pneumonia-project/data/`. Skips with a message if it isn't there
yet, and skips the unzip if the data is already unpacked, so the cell is safe to re-run.

In [ ]:
import zipfile

if DATA_ROOT.is_dir():
    print(f"dataset: already unpacked at {DATA_ROOT}")
elif not DATA_ZIP.is_file():
    # Not an error. Until the zip is uploaded, every cell above still works and only the two
    # cells from here down need the data.
    print(f"dataset: {DATA_ZIP} not on Drive yet - skipping")
else:
    DATA_DIR.mkdir(parents=True, exist_ok=True)
    # Reading one large file sequentially off Drive and writing to the local session disk is
    # something Drive handles well - which is the entire reason the dataset is stored zipped
    # rather than as ~5,800 separate files. Expect a couple of minutes.
    with zipfile.ZipFile(DATA_ZIP) as archive:
        # extractall() unpacks every member and recreates the folder structure recorded inside
        # the zip, so the archive has to contain chest_xray/train/... rather than loose files.
        # If DATA_ROOT doesn't exist afterwards, the zip was built from the wrong folder level.
        archive.extractall(DATA_DIR)
    print(f"dataset: unpacked to {DATA_ROOT}")

## 10. Verify the extraction

Counts images per split and class against `data/dataset_manifest.json` — the dataset
description committed with the data — and checks for the two packaging faults that would
otherwise stay invisible until training. Raises on any mismatch.

In [ ]:
import json

MANIFEST_PATH = REPO_DIR / "data" / "dataset_manifest.json"

if not DATA_ROOT.is_dir():
    print(f"dataset: nothing at {DATA_ROOT} - run the cell above first")
elif not MANIFEST_PATH.is_file():
    # The manifest is written by whoever prepares the dataset and committed with it. Until it
    # exists there is nothing authoritative to check against, and hardcoding the numbers here
    # is exactly what this cell is built to avoid - they would silently go stale.
    print(f"dataset: {MANIFEST_PATH} not in the repo yet - skipping the count check")
else:
    manifest = json.loads(MANIFEST_PATH.read_text())
    extensions = {extension.lower() for extension in manifest["image_extensions"]}

    problems = []
    total = 0
    # counts is {split: {class: number}}, so the nested loop follows whatever splits and classes
    # the manifest declares - the notebook never assumes train/val/test or NORMAL/PNEUMONIA.
    for split, per_class in manifest["counts"].items():
        for cls, expected in per_class.items():
            folder = DATA_ROOT / split / cls
            # rglob("*") walks the folder and everything under it; the extension filter drops
            # .DS_Store and any other stray non-image file that travelled inside the zip.
            found = sum(1 for p in folder.rglob("*") if p.suffix.lower() in extensions)
            total += found
            if found != expected:
                problems.append(f"{split}/{cls}: found {found}, manifest says {expected}")
            print(f"{split:8s} {cls:10s} {found:5d}  {'ok' if found == expected else 'MISMATCH'}")
    # Summed here rather than read from the manifest, so the total cannot contradict its parts.
    print(f"{'total':19s} {total:5d}")

    # Two faults that are properties of a badly built zip rather than of the dataset, so they
    # stay hardcoded: __MACOSX (metadata macOS adds when zipping) and a second chest_xray nested
    # inside the first. The nested copy would not show up in the counts above at all - those
    # only look under DATA_ROOT/<split> - which is why it is checked separately.
    for junk in (DATA_ROOT / "__MACOSX", DATA_ROOT / manifest["top_level_folder"], DATA_DIR / "__MACOSX"):
        if junk.exists():
            problems.append(f"{junk} exists - the zip was not cleaned")

    # Raises rather than warns: the manifest is the agreed description of the dataset, so a
    # mismatch means the zip on Drive is not the dataset this repo documents.
    assert not problems, "dataset does not match the manifest:\n  " + "\n  ".join(problems)
    print("dataset matches the manifest")

## 11. Run one experiment - the E01 pilot

`run_experiment.py --id` runs one config end to end: train, then evaluate. An `E` run chooses its
decision threshold on validation, scores the official test split, measures Grad-CAM stability on
the fixed test subset, draws its three figures and writes `results/metrics/E##_metrics.json` -
that file last, so it only ever exists complete. A `T` run scores validation only.

**The pilot:** E01 once, checked against the pass criteria in the plan (header, files, audit 19 of
97, timing) before the sweep starts. **If the session drops, run the same command again** - it
resumes from the last checkpoint. A finished experiment is skipped; `--reevaluate` redoes only its
evaluation (after a fix there), `--force` retrains it from scratch.

`-u` makes Python write output as it happens; without it `print()` output and tracebacks sit in
a buffer until the process exits.

In [ ]:
!python -u run_experiment.py --id E01 --workers 2

# After a fix in evaluation or stability:   !python -u run_experiment.py --id E01 --workers 2 --reevaluate
# After a fix in training:                  !python -u run_experiment.py --id E01 --workers 2 --force
# Smoke tests go to SMOKE_ROOT, never the Drive root: a short run under the real root would mark
# the experiment finished, and the full run would then be skipped as already done.
# !python -u run_experiment.py --id E01 --max-epochs 2 --workers 2 --output-root /content/smoke


## 12. What landed on Drive

The audit reads the title stored inside every figure and checks it against the config that
should have produced it, so a stale or mislabelled figure is caught here rather than in the
report. The audit expects 97 figures once every run is done: three per experiment and training curves for
each of the sixteen tuning runs (curves from extra seeds are not counted).

In [ ]:
!python -m src.figures --audit
# Summary figures and results/summary_table.md from the results files found so far (seed 42).
!python -m src.figures --report

for folder in (RESULTS_METRICS, RESULTS_FIGURES, CHECKPOINTS):
    files = sorted(path for path in folder.rglob("*") if path.is_file())
    print(f"\n{folder} - {len(files)} files")
    for path in files[:10]:
        # st_size is in bytes; /1e6 gives MB, which is the useful unit for checkpoints.
        print(f"   {path.relative_to(folder).as_posix():45s} {path.stat().st_size / 1e6:6.1f} MB")


## Next steps

The tuning phase, sixteen validation-only runs that fixed the recipe (done):

    !python -u run_all.py --tuning --workers 2

and **the sweep, once the E01 pilot has passed** - all 27 experiments, seed 42:

    !python -u run_all.py --workers 2

Its first line should read `E01: already done, skipping`. It skips runs that already have
results, so a dropped session costs one run rather than the whole sweep; a run that fails is
reported and the sweep continues, and rerunning the same cell retries it. Expect 2-3 sessions.
After each one, run section 12 (audit + report). Results, figures and checkpoints go to the Drive
paths set in section 3.